# Lab 12: Ingestion at scale

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /home/jsepu/LanternCS2 | LLM mode: replay


Set `ONLINE = True` (and put your team name and Northeastern email in `H`) to call EDGAR for real. Offline, the slide code below reads structure samples from `data/edgar_fixture/` (fictional values).

In [2]:
ONLINE = False
import requests
class _FixtureResponse:
    def __init__(self, path): self.path = path
    def json(self): return json.load(open(self.path))
def _offline_get(url, headers=None, timeout=None):
    if "submissions/CIK" in url: return _FixtureResponse("data/edgar_fixture/submissions_SAMPLE.json")
    if "companyfacts/CIK" in url: return _FixtureResponse("data/edgar_fixture/companyfacts_SAMPLE.json")
    raise RuntimeError(f"offline: no fixture for {url}")
if not ONLINE:
    requests.get = _offline_get

In [3]:
# From the slide: The submissions API: find exactly the filings you need
import requests
H = {"User-Agent": "NEU-DAMG7245-codeofduty3 sepulveda.ju@northeastern.edu"}

def submissions(cik: str) -> dict:
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    return requests.get(url, headers=H, timeout=30).json()

def earnings_8ks(cik: str, n: int = 4) -> list[dict]:
    r = submissions(cik)["filings"]["recent"]  # columns
    rows = zip(r["form"], r["items"], r["filingDate"],
               r["accessionNumber"], r["primaryDocument"])
    out = [dict(form=f, items=i, date=d, accn=a, doc=p)
           for f, i, d, a, p in rows
           if f == "8-K" and "2.02" in i.split(",")]
    return out[:n]         # newest first

In [4]:
earnings_8ks("0000000001")

[{'form': '8-K',
  'items': '2.02,9.01',
  'date': '2026-08-05',
  'accn': '0000000001-26-000041',
  'doc': 'smpa-8k_20260805.htm'},
 {'form': '8-K',
  'items': '2.02,9.01',
  'date': '2026-05-06',
  'accn': '0000000001-26-000022',
  'doc': 'smpa-8k_20260506.htm'}]

In [5]:
# From the slide: companyfacts at scale: facts, not documents
import pandas as pd
KEEP = ("start", "end", "val", "fy", "fp", "form",
        "accn", "frame")

def facts_frame(cik: str) -> pd.DataFrame:
    url = ("https://data.sec.gov/api/xbrl/"
           f"companyfacts/CIK{cik}.json")
    j = requests.get(url, headers=H, timeout=30).json()
    rows = []
    for tax, concepts in j["facts"].items():
        for concept, body in concepts.items():
            for unit, obs in body["units"].items():
                for o in obs:
                    r = {k: o.get(k) for k in KEEP}
                    r.update(cik=cik, taxonomy=tax,
                             concept=concept, unit=unit)
                    rows.append(r)
    return pd.DataFrame(rows)

In [6]:
facts_frame("0000000001")[["concept", "unit", "end", "val", "fp", "form"]]

,concept,unit,end,val,fp,form
0,EntityCommonStockSharesOutstanding,shares,2026-07-31,61000000,Q2,10-Q
1,Revenues,USD,2025-12-31,820000000,FY,10-K
2,Revenues,USD,2026-06-30,430000000,Q2,10-Q
3,NetIncomeLoss,USD,2025-12-31,71000000,FY,10-K


The data room arrives as CS1-style records (`data/dataroom/deal_a/export/records.jsonl`), already in `lantern/2.0`. Value-tiered routing decides which path each source takes:

In [7]:
print(llm.P["ingest"]["routing"])
rec = sd.records("deal_a")[0]
print({k: rec[k] for k in ("schema", "doc_id", "deal_id", "source_type", "tier", "page", "block_id", "citation")})

{'comps': 'xbrl+docling_items', 'dataroom': 'cs1_full+docling+ocr', 'web': 'html_text'}
{'schema': 'lantern/2.0', 'doc_id': 'DR-deal_a-001', 'deal_id': 'deal_a', 'source_type': 'dataroom', 'tier': 4, 'page': 1, 'block_id': 'p0001_b001', 'citation': {'kind': 'pdf_bbox', 'doc_id': 'DR-deal_a-001', 'page': 1, 'bbox': [72.0, 91.5, 528.8, 143.0]}}


In [8]:
# From the slide: The synthetic web: crawl from the sitemap, inside an allowlist
import hashlib, xml.etree.ElementTree as ET
from urllib.parse import urlparse
from bs4 import BeautifulSoup
SM = "{http://www.sitemaps.org/schemas/sitemap/0.9}"

def sitemap_urls(xml_text):
    return [l.text.strip()
            for l in ET.fromstring(xml_text).iter(f"{SM}loc")]

def allowed(url, hosts):            # refuse everything else
    u = urlparse(url)
    return u.scheme == "https" and u.hostname in hosts

def web_record(url, html, deal_id):
    soup = BeautifulSoup(html, "html.parser")
    for t in soup(["script", "style", "nav"]):
        t.decompose()
    text = " ".join(soup.get_text(" ").split())
    sha = hashlib.sha256(html.encode()).hexdigest()
    return {"schema": "lantern/2.0", "doc_id": f"WEB-{sha[:8]}",
            "deal_id": deal_id, "source_type": "synthetic_web",
            "tier": 5, "text": text, "source_path": url,
            "citation": {"kind": "web_span", "url": url},
            "lineage": {"parents": [], "source_sha256": sha},
            "ingest_flags": []}

In [9]:
# Offline, URLs on the course host map to files in data/web_site/
from urllib import robotparser
HOSTS = {llm.P["web"]["host"]}
urls = sitemap_urls(open("data/web_site/sitemap.xml").read())
rp = robotparser.RobotFileParser(); rp.parse(open("data/web_site/robots.txt").read().splitlines())
web = []
for u in urls:
    if not allowed(u, HOSTS) or not rp.can_fetch("LANTERN", u):
        print("refused:", u); continue
    html = open("data/web_site/" + urlparse(u).path.lstrip("/")).read()
    web.append(web_record(u, html, "deal_a"))
print(len(web), "web records;", web[0]["doc_id"], "tier", web[0]["tier"], "|", web[0]["text"][:60])

refused: https://not-allowed.example.com/tracker.html
3 web records; WEB-ed15cb2c tier 5 | About Wren About Wren Freight Software Wren builds dispatch 


**Exercise 12.** Point `earnings_8ks` at your CS1 company with `ONLINE = True`. How many 8-K filings report Item 2.02 in the last year?

In [10]:
# Exercise 12: CS1 company = Akamai Technologies (AKAM), CIK 0001086222
import importlib, datetime as dt
ONLINE = True
importlib.reload(requests)   # undo the offline fixture patch from the setup cell
H = {"User-Agent": "NEU-DAMG7245-codeofduty3 sepulveda.ju@northeastern.edu"}

AKAM = "0001086222"
all_202 = earnings_8ks(AKAM, n=100)
cutoff = (dt.date.today() - dt.timedelta(days=365)).isoformat()
last_year = [x for x in all_202 if x["date"] >= cutoff]
for x in last_year:
    print(x["date"], x["items"], x["accn"], x["doc"])
print(f"\n{len(last_year)} Item 2.02 8-Ks since {cutoff}")

2026-08-06 2.02,9.01 0001086222-26-000083 akam-20260806.htm
2026-05-07 2.02,9.01 0001086222-26-000054 akam-20260507.htm
2026-02-19 2.02,9.01 0001086222-26-000017 akam-20260219.htm
2025-11-06 2.02,9.01 0001086222-25-000252 akam-20251106.htm

4 Item 2.02 8-Ks since 2025-10-24


**Discussion (tutorial Ex. 12).** Shortlist 12 candidate comps for Akamai (comparable #1), pull companyfacts for each, cut to 6–10, and compare which XBRL tags they use for the same idea. Stretch: fetch concurrently while staying under 10 requests per second.

In [11]:
# Shortlist: Akamai plus 12 candidates across CDN/edge, network security and cloud infrastructure
import time, threading
from concurrent.futures import ThreadPoolExecutor

_lock, _last = threading.Lock(), [0.0]
def polite_get(url):                      # stretch: shared limiter, at most ~8 requests per second across threads
    with _lock:
        wait = 0.125 - (time.monotonic() - _last[0])
        if wait > 0: time.sleep(wait)
        _last[0] = time.monotonic()
    return requests.get(url, headers=H, timeout=30).json()

tickers = polite_get("https://www.sec.gov/files/company_tickers.json")
CIK = {v["ticker"]: str(v["cik_str"]).zfill(10) for v in tickers.values()}
CANDIDATES = ["AKAM", "NET", "FSLY", "FFIV", "DOCN", "ZS", "OKTA", "FTNT", "PANW", "CRWD", "NTNX", "GDDY", "TWLO"]
REV_TAGS = ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax", "RevenueFromContractWithCustomerIncludingAssessedTax"]

def profile(t):
    sub = polite_get(f"https://data.sec.gov/submissions/CIK{CIK[t]}.json")
    gaap = polite_get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK[t]}.json")["facts"].get("us-gaap", {})
    annual = [(tag, o) for tag in REV_TAGS for o in gaap.get(tag, {}).get("units", {}).get("USD", [])
              if o.get("form") == "10-K" and o.get("fp") == "FY" and len(o.get("frame", "")) == 6]   # CYyyyy duration frames
    tag, latest = max(annual, key=lambda x: x[1]["end"]) if annual else (None, {})
    return {"ticker": t, "name": sub["name"], "sic": sub.get("sic"), "sic_desc": sub.get("sicDescription"),
            "hq": sub.get("addresses", {}).get("business", {}).get("stateOrCountry"), "fye": sub.get("fiscalYearEnd"),
            "rev_tag": tag, "revenue_bn": round(latest.get("val", 0) / 1e9, 2), "rev_frame": latest.get("frame"),
            "concepts": set(gaap)}

with ThreadPoolExecutor(max_workers=4) as pool:
    comps_raw = {p["ticker"]: p for p in pool.map(profile, CANDIDATES)}
pd.DataFrame(comps_raw.values()).drop(columns="concepts")

,ticker,name,sic,sic_desc,hq,fye,rev_tag,revenue_bn,rev_frame
0,AKAM,AKAMAI TECHNOLOGIES INC,7389,"Services-Business Services, NEC",MA,1231,RevenueFromContractWithCustomerIncludingAssess...,4.21,CY2025
1,NET,"Cloudflare, Inc.",7372,Services-Prepackaged Software,CA,1231,RevenueFromContractWithCustomerExcludingAssess...,2.17,CY2025
2,FSLY,"Fastly, Inc.",7372,Services-Prepackaged Software,CA,1231,RevenueFromContractWithCustomerIncludingAssess...,0.62,CY2025
3,FFIV,"F5, INC.",3576,Computer Communications Equipment,WA,0930,RevenueFromContractWithCustomerExcludingAssess...,3.09,CY2025
4,DOCN,"DigitalOcean Holdings, Inc.",7370,"Services-Computer Programming, Data Processing...",CO,1231,RevenueFromContractWithCustomerExcludingAssess...,0.90,CY2025
5,ZS,"Zscaler, Inc.",7371,Services-Computer Programming Services,CA,0731,RevenueFromContractWithCustomerExcludingAssess...,3.35,CY2026
6,OKTA,"Okta, Inc.",7372,Services-Prepackaged Software,CA,0131,RevenueFromContractWithCustomerExcludingAssess...,2.92,CY2025
7,FTNT,"Fortinet, Inc.",3577,"Computer Peripheral Equipment, NEC",CA,1231,RevenueFromContractWithCustomerExcludingAssess...,6.80,CY2025
8,PANW,Palo Alto Networks Inc,3577,"Computer Peripheral Equipment, NEC",CA,0731,RevenueFromContractWithCustomerExcludingAssess...,11.48,CY2026
9,CRWD,"CrowdStrike Holdings, Inc.",7372,Services-Prepackaged Software,TX,0131,RevenueFromContractWithCustomerIncludingAssess...,4.81,CY2025


In [12]:
# Cut to 8 (Akamai + 7) by business model, size and geography; then compare tags for the same idea
KEEP = ["AKAM", "NET", "FSLY", "FFIV", "ZS", "FTNT", "PANW", "DOCN"]
common = set.intersection(*(comps_raw[t]["concepts"] for t in KEEP))
print(len(common), "us-gaap concepts reported by all", len(KEEP), "comps")
IDEAS = {"revenue": REV_TAGS,
         "cost of revenue": ["CostOfRevenue", "CostOfGoodsAndServicesSold"],
         "gross profit": ["GrossProfit"],
         "long-term debt": ["LongTermDebt", "LongTermDebtNoncurrent", "ConvertibleNotesPayableCurrent"],
         "D&A": ["DepreciationDepletionAndAmortization", "DepreciationAndAmortization"],
         "operating income": ["OperatingIncomeLoss"], "net income": ["NetIncomeLoss"],
         "cash": ["CashAndCashEquivalentsAtCarryingValue"], "SBC": ["ShareBasedCompensation"],
         "CFO": ["NetCashProvidedByUsedInOperatingActivities"], "capex": ["PaymentsToAcquirePropertyPlantAndEquipment"]}
rows = [{"idea": idea, "tag": tag, "reported_by": "ALL" if all(tag in comps_raw[t]["concepts"] for t in KEEP)
         else " ".join(t for t in KEEP if tag in comps_raw[t]["concepts"]) or "-"}
        for idea, tags in IDEAS.items() for tag in tags]
pd.DataFrame(rows)

90 us-gaap concepts reported by all 8 comps


,idea,tag,reported_by
0,revenue,Revenues,AKAM ZS PANW
1,revenue,RevenueFromContractWithCustomerExcludingAssess...,NET FFIV ZS FTNT PANW DOCN
2,revenue,RevenueFromContractWithCustomerIncludingAssess...,AKAM FSLY FFIV
3,cost of revenue,CostOfRevenue,AKAM NET FSLY ZS DOCN
4,cost of revenue,CostOfGoodsAndServicesSold,FFIV ZS FTNT PANW
5,gross profit,GrossProfit,NET FSLY FFIV ZS FTNT PANW DOCN
6,long-term debt,LongTermDebt,FSLY ZS FTNT PANW DOCN
7,long-term debt,LongTermDebtNoncurrent,FSLY FFIV FTNT DOCN
8,long-term debt,ConvertibleNotesPayableCurrent,AKAM ZS
9,D&A,DepreciationDepletionAndAmortization,ALL


**Comps (8):** AKAM, NET, FSLY, FFIV, ZS, FTNT, PANW, DOCN: same mix of edge delivery, security and cloud as Akamai. Dropped GDDY, TWLO, NTNX, OKTA, CRWD (different business model).

**Reported by all:** operating income, net income, D&A, cash, SBC, operating cash flow, capex, R&D, S&M, G&A.

**Same idea, different tags:** revenue (AKAM/FSLY use `...IncludingAssessedTax`, the rest `...ExcludingAssessedTax`); cost of revenue (`CostOfRevenue` vs `CostOfGoodsAndServicesSold`); AKAM has no `GrossProfit` tag.

**Large comp, PANW (~$11.5bn):** it competes directly with Akamai's security business for the same enterprise customers. Use the median so its size doesn't skew the multiples.